In [1]:
import os 
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import GroupShuffleSplit
from PIL import Image
from torch.utils.data import Dataset, DataLoader, random_split
import torch
import torch.nn as nn
import torchvision.transforms as transforms
import torchvision.models as models
import torchvision.transforms as T
from torchmetrics.classification import MulticlassRecall, MulticlassAccuracy
from tqdm import tqdm
from collections import Counter

from Data_Utility.lookup_size import lookup_size_from_excel
from Data_Utility.dataset import PollenFolderWithSizeDataset

from models.basemodel import CNNWithSizeMLP

from engine.train_epoch import train_epoch
from engine.train_model import train_model
from engine.eval_epoch import eval_epoch
from engine.plots_loss_accuracy import plot_training_history
from engine.plot_confmat import plot_confusion_matrix
from engine.data_setup import setup_datasets

print("All libraries imported successfully!")

All libraries imported successfully!


In [2]:
test_dir = '/home/na1488tr-s/Bachelor_Project_Statistics/Data/Size-data/TestData_sensi'
train_dir = '/home/na1488tr-s/Bachelor_Project_Statistics/Data/Size-data/TrainData_sensi'
test_excel_path = '/home/na1488tr-s/Bachelor_Project_Statistics/Data/Size-data/Size_features/size_data.xlsx'


def collect_filenames(root_dir):
    """
    Collect all image filenames recursively from root_dir.
    Returns list of filenames (without path).
    """
    filenames = []
    for species in os.listdir(root_dir):
        species_dir = os.path.join(root_dir, species)
        if not os.path.isdir(species_dir):
            continue

        for fn in os.listdir(species_dir):
            if fn.lower().endswith(('.png', '.jpg', '.jpeg')):
                filenames.append(fn)
    return filenames


# Collect filenames
train_files = collect_filenames(train_dir)
test_files  = collect_filenames(test_dir)

print(f"Train images: {len(train_files)}")
print(f"Test images: {len(test_files)}")

Train images: 8078
Test images: 1922


In [3]:
# Convert to pandas DataFrame
train_df = pd.DataFrame({'filename': train_files})
test_df  = pd.DataFrame({'filename': test_files})

# Extract flower ID (before underscore)
train_df['flower_id'] = train_df['filename'].str.split(' ').str[-1].str.split('_').str[0]
test_df['flower_id']  = test_df['filename'].str.split(' ').str[-1].str.split('_').str[0]

print(train_df.head())

                             filename flower_id
0  Tussilago farfara 1000328_3192.png   1000328
1  Tussilago farfara 1005387_1866.png   1005387
2  Tussilago farfara 1001625_1653.png   1001625
3  Tussilago farfara 1000328_1189.png   1000328
4   Tussilago farfara 1004811_656.png   1004811


In [4]:
train_flower_ids = set(train_df['flower_id'])
test_flower_ids  = set(test_df['flower_id'])

overlap = train_flower_ids.intersection(test_flower_ids)

print(f"Unique flower IDs in train: {len(train_flower_ids)}")
print(f"Unique flower IDs in test: {len(test_flower_ids)}")
print(f"Overlap in flower IDs: {len(overlap)}")

print(f"\n Overlapping flower IDs: {overlap}")


Unique flower IDs in train: 49
Unique flower IDs in test: 11
Overlap in flower IDs: 0

 Overlapping flower IDs: set()


In [5]:
def extract_species_from_filename(fn):
    # remove trailing "1234567_89.png"
    return " ".join(fn.split(" ")[:-1])

train_df["species"] = train_df["filename"].apply(extract_species_from_filename)
test_df["species"] = test_df["filename"].apply(extract_species_from_filename)

print(train_df.head())
print(test_df.head())

                             filename flower_id            species
0  Tussilago farfara 1000328_3192.png   1000328  Tussilago farfara
1  Tussilago farfara 1005387_1866.png   1005387  Tussilago farfara
2  Tussilago farfara 1001625_1653.png   1001625  Tussilago farfara
3  Tussilago farfara 1000328_1189.png   1000328  Tussilago farfara
4   Tussilago farfara 1004811_656.png   1004811  Tussilago farfara
                             filename flower_id            species
0    Tussilago farfara 1005388_97.png   1005388  Tussilago farfara
1  Tussilago farfara 1005388_1993.png   1005388  Tussilago farfara
2  Tussilago farfara 1005388_1175.png   1005388  Tussilago farfara
3  Tussilago farfara 1005388_2027.png   1005388  Tussilago farfara
4   Tussilago farfara 1005388_975.png   1005388  Tussilago farfara


In [6]:
flowers_per_species = (
    train_df.groupby("species")["flower_id"]
    .nunique()
    .sort_values(ascending=False)
)
print("\nNumber of unique flowers per species in training set:")
print(flowers_per_species)


Number of unique flowers per species in training set:
species
Brassica napus             8
Cichorium intybus          8
Bellis perennis            6
Capsella bursa-pastoris    5
Tussilago farfara          5
Crepis capillaris          4
Hieracium umbellatum       4
Hypochaeris radicata       4
Tragopogon pratensis       3
Sonchus arvensis           2
Name: flower_id, dtype: int64


In [7]:
flowers_per_species = (
    test_df.groupby("species")["flower_id"]
    .nunique()
    .sort_values(ascending=False)
)
print("\nNumber of unique flowers per species in test set:")
print(flowers_per_species)


Number of unique flowers per species in test set:
species
Brassica napus             2
Bellis perennis            1
Capsella bursa-pastoris    1
Cichorium intybus          1
Crepis capillaris          1
Hieracium umbellatum       1
Hypochaeris radicata       1
Sonchus arvensis           1
Tragopogon pratensis       1
Tussilago farfara          1
Name: flower_id, dtype: int64


In [8]:
size_df = pd.read_excel(test_excel_path)

size_df["flower_id"] = (
    size_df["ping_name"]
    .astype(str)
    .str.split(" ").str[-1]
    .str.split("_").str[0]
)

size_df["species"] = size_df["anyname"]

print(size_df[["ping_name", "species", "flower_id", "majoraxis", "minoraxis"]].head())

                          ping_name          species flower_id  majoraxis  \
0  Bellis perennis 1001572_1013.png  Bellis perennis   1001572    26.1271   
1  Bellis perennis 1001572_1019.png  Bellis perennis   1001572    23.2606   
2  Bellis perennis 1001572_1022.png  Bellis perennis   1001572    25.6662   
3  Bellis perennis 1001572_1039.png  Bellis perennis   1001572    25.2603   
4  Bellis perennis 1001572_1057.png  Bellis perennis   1001572    28.9809   

   minoraxis  
0    24.5456  
1    22.8500  
2    24.1594  
3    24.8495  
4    25.3248  


In [9]:
train_size_df = size_df[size_df["ping_name"].isin(train_files)].copy()

print("Rows in Excel matching train images:", len(train_size_df))

Rows in Excel matching train images: 7922


In [10]:
test_size_df = size_df[size_df["ping_name"].isin(test_files)].copy()

print("Rows in Excel matching test images:", len(test_size_df))

Rows in Excel matching test images: 1922


In [11]:
species_name = "Crepis capillaris"

train_species_df = train_df[train_df["species"] == species_name].copy()
print("Train images for species:", len(train_species_df))
print(train_species_df.head())

Train images for species: 800
                                filename flower_id            species
3157    Crepis capillaris 1010538_81.png   1010538  Crepis capillaris
3158  Crepis capillaris 1009512_4057.png   1009512  Crepis capillaris
3159   Crepis capillaris 1009502_109.png   1009502  Crepis capillaris
3160  Crepis capillaris 1009502_1546.png   1009502  Crepis capillaris
3161  Crepis capillaris 1000322_3204.png   1000322  Crepis capillaris


In [12]:
excel_species_df = size_df[size_df["ping_name"].isin(train_species_df["filename"])].copy()

print("Excel rows matching train images for species:", len(excel_species_df))
print(excel_species_df.head())

Excel rows matching train images for species: 799
      acceptid            anyname      family  \
3993    220099  Crepis capillaris  Asteraceae   
3994    220099  Crepis capillaris  Asteraceae   
3995    220099  Crepis capillaris  Asteraceae   
3996    220099  Crepis capillaris  Asteraceae   
3997    220099  Crepis capillaris  Asteraceae   

                               ping_name  majoraxis  minoraxis flower_id  \
3993  Crepis capillaris 1000322_1156.png    31.6935    27.8145   1000322   
3994  Crepis capillaris 1000322_1186.png    31.7569    29.6723   1000322   
3995  Crepis capillaris 1000322_1188.png    31.4411    30.0027   1000322   
3996    Crepis capillaris 1000322_12.png    38.0422    25.3304   1000322   
3997  Crepis capillaris 1000322_1215.png    33.6671    29.0045   1000322   

                species  
3993  Crepis capillaris  
3994  Crepis capillaris  
3995  Crepis capillaris  
3996  Crepis capillaris  
3997  Crepis capillaris  


In [13]:
excel_species_df["flower_id"] = (
    excel_species_df["ping_name"].str.split(" ").str[-1].str.split("_").str[0]
)

In [14]:
train_counts = (
    train_species_df.groupby("flower_id")
    .size()
    .reset_index(name="n_train_images")
)

excel_counts = (
    excel_species_df.groupby("flower_id")
    .size()
    .reset_index(name="n_excel_matches")
)

compare_df = train_counts.merge(excel_counts, on="flower_id", how="left")
compare_df["n_excel_matches"] = compare_df["n_excel_matches"].fillna(0).astype(int)
compare_df["all_images_exist_in_excel"] = (
    compare_df["n_train_images"] == compare_df["n_excel_matches"]
)

print(compare_df.sort_values("flower_id"))

  flower_id  n_train_images  n_excel_matches  all_images_exist_in_excel
0   1000322             200              199                      False
1   1009502             200              200                       True
2   1009512             200              200                       True
3   1010538             200              200                       True


In [15]:
complete_flower_ids = compare_df.loc[
    compare_df["all_images_exist_in_excel"], "flower_id"
].tolist()

print("Flower IDs with all train images present in Excel:")
print(complete_flower_ids)

Flower IDs with all train images present in Excel:
['1009502', '1009512', '1010538']


In [16]:


train_dataset, val_dataset, test_dataset, class_to_idx, idx_to_class,classes = setup_datasets(
    val_bool=True,
    aug_bool=True
)
num_classes = len(classes)

Total samples: 9844
No missing size data found. All rows have valid majoraxis and minoraxis values in Excel file.

Initialized dataset from '/home/na1488tr-s/Bachelor_Project_Statistics/Data/Size-data/Sorted_224_sizeTrain' with 7922 samples.

=========== /home/na1488tr-s/Bachelor_Project_Statistics/Data/Size-data/Sorted_224_sizeTrain ===========
fill_missing_bool: False
bootstrap_impute_bool: False
Total scanned images: 8078
   Final dataset size:   7922
   Missing size found:   156

Missing size by species:
      Bellis perennis: 2 
      Brassica napus: 5 
      Crepis capillaris: 1 
      Hieracium umbellatum: 65 
      Hypochaeris radicata: 83 
Dropped missing:      156


=== Augmented-to-max dataset built from '/home/na1488tr-s/Bachelor_Project_Statistics/Data/Size-data/Sorted_224_sizeTrain' ===
Original samples kept: 7922
Extra augmented samples added: 778
Final dataset size: 8700
Target per-class size (n_max): 870
Per-class original sizes:
  Bellis perennis: 856 (add 14)
  Brass